# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'https://raw.githubusercontent.com/jasontpv/DS2002FA26/main/notebooks/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
    print('good')
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
    print('bad')
cat.head()

good


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


In [2]:
print('Rows:', len(cat))
print('Columns:', len(cat.columns), ' | ', cat.columns.tolist())
print('Duplicates:', cat.duplicated().sum())
print('Columns that arrived as text:', [c for c in cat.columns if not pd.api.types.is_numeric_dtype(cat[c])])
print('Nulls per column:')
print(cat.isnull().sum()[lambda s: s > 0].to_string())

Rows: 45
Columns: 7  |  ['sku', 'product_name', 'category', 'subcategory', 'unit_cost', 'supplier', 'package_size']
Duplicates: 0
Columns that arrived as text: ['sku', 'product_name', 'category', 'subcategory', 'supplier', 'package_size']
Nulls per column:
subcategory    13
unit_cost       5


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
pop_products = pd.DataFrame(cat[cat['product_name'].str.lower().str.contains('pop', na=False)]) #filters produst on product_name containing pop into new dataframe

print("Count:", len(pop_products))
display(pop_products)

Count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack




```
# This is formatted as code
```

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
strawberry_pop_tarts = pop_products[pop_products['product_name'].str.contains('strawberry', case=False, na=False)] #create df with pop-tart products that contain strawberry in the name
print("Strawberry Pop-Tarts:")
display(strawberry_pop_tarts[['sku', 'product_name']])

print('\n')

other_pop_tarts = pop_products[~pop_products['product_name'].str.contains('strawberry', case=False, na=False)] #create df with pop-tart products not containing strawberry in the name
print("Other Pop-Tarts (must stay separate):")
display(other_pop_tarts[['sku', 'product_name']])

Strawberry Pop-Tarts:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry




Other Pop-Tarts (must stay separate):


,sku,product_name


**Pop-Tart rows that are a different product:** _None; all pop-tart rows are strawberry_

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
CANON = {                                 #strawberry poptart sku map
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

def consolidate(df):                                # function to apply sku map
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
    return out

consolidated = consolidate(cat)
assert len(consolidated) == len(cat), \
    'consolidation must not change length'
consolidated.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,POPTART-STRAW
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
print("Categories BEFORE normalization:")
print(consolidated['category'].value_counts(dropna=False).sort_index())       # prints categories before being noramlized

consolidated['category'] = consolidated['category'].str.strip().str.title()   # normalizes categories by stripping and making title case

print("\nCategories AFTER normalization:")
print(consolidated['category'].value_counts(dropna=False).sort_index())       # prints normalized categories and counts

Categories BEFORE normalization:
category
ALCOHOL                1
Alcoholic Beverages    1
Baby                   2
Baby Products          1
Breakfast              2
Breakfast Foods        1
ELECTRONICS            2
EMERGENCY              3
ENTERTAINMENT          3
Electronics            1
Emergency              5
Emergency Supplies     3
GROCERY                3
Groceries              1
Grocery                1
HOUSEHOLD              1
Household              1
Snacks                 1
baby                   1
breakfast              1
electronics            1
emergency              1
entertainment          2
grocery                1
health                 2
household              2
snacks                 1
Name: count, dtype: int64

Categories AFTER normalization:
category
Alcohol                1
Alcoholic Beverages    1
Baby                   3
Baby Products          1
Breakfast              3
Breakfast Foods        1
Electronics            4
Emergency              9
Emergency Suppl

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
import re

def normalize_name(name):
    name = name.lower().strip()              # Lowercase and strip leading/trailing whitespace
    name = re.sub(r'[^\w\s]', '', name)      # Remove punctuation
    return ' '.join(sorted(name.split()))    # Sort words alphabetically

consolidated['normalized_name'] = consolidated['product_name'].apply(normalize_name)    # Create a normalized name column

# Group by normalized name and find groups with multiple SKUs
candidates = (consolidated.groupby('normalized_name').filter(lambda group: group['sku'].nunique() > 1).sort_values('normalized_name'))

display(candidates[['normalized_name', 'sku', 'product_name', 'category']])

,normalized_name,sku,product_name,category
12,24pack beer light,BEER-24,Light Beer (24-pack),Alcoholic Beverages
13,24pack beer light,LTBEER24,Light Beer (24-pack),Alcohol
8,4pack aa batteries,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies
9,4pack aa batteries,BATT_AA_4PK,AA Batteries 4-Pack,Emergency
14,board game monopoly,BG-MONO,Monopoly Board Game,Entertainment
16,board game monopoly,GAME-MONO,Board Game - Monopoly,Entertainment
15,board game monopoly,MONOPOLY-01,Monopoly Board Game,Entertainment
23,bread white,BREAD-WH,WHITE BREAD,Grocery
24,bread white,WHITEBREAD,WHITE BREAD,Grocery
26,butter peanut,PNUTBTR-16,Peanut Butter,Grocery


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
assert consolidated['sku'].notna().all(), \
    'Canonical SKU column contains nulls'

assert len(consolidated) == len(cat), \
    'Consolidation must not change the number of rows'

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Light Beer (24-packs) and AA Batteries 4-pack are fragemented, and I would group them by consolidating the sku for them to LTBEER24 and BAT-AA4, respectively.

b) Consolidation stops being a data-cleaning task when deciding whether two similar records represent the same product for business purposes, rather than simply having differently formatted names or SKUs. For example, there are emergency candles and candles (emergency); are the latter meant as backup stock where the emergency is the stores and the former meant for customers as candles for  personal emergencies? I would ask who ever created the inventory list, like an inventory manager, what the proper assignment is.